# CBB Analytics REST API: sample Python client

Docs: https://rest.cbbanalytics.com/api-docs/

Building a scheduled pipeline (backfill, daily sync, portal tracking)? Read `PIPELINE_GUIDE.md` and use `scripts/cbb_sync.py`; this notebook covers calling the API.

| Version | Fields | Response shape |
|---|---|---|
| **v3** (recommended) | All current fields | `{ response: { meta, data } }`; `meta` carries `count`, `limit`, `nextCursor`, `hasMore` and more |
| **v2** (legacy) | All current fields | Bare JSON array |
| **v1** (legacy) | A frozen field set | Bare JSON array |

**Cursor pagination:** send the first request without `after`, then send each response's `meta.nextCursor` back unchanged as `after` until it's missing. Don't build or edit cursors yourself, and don't send `sortBy` (v3 rejects it).

Set `API_KEY` in section (A), then run the cells in order. Section (D) pulls real data, and every page counts against your key's request limits.

## (A) Setup

In [ ]:
# list any missing packages and how to install them
import importlib.util
missing = [pkg for pkg in ['pandas', 'requests'] if importlib.util.find_spec(pkg) is None]
if missing:
    print(f'Missing packages; run: pip install {" ".join(missing)}')

In [ ]:
# system imports
import time
from datetime import date, timedelta

# third party
import pandas as pd
import requests

# connection settings; set your key here
API_KEY = 'YOUR_API_KEY_HERE'
HOST = 'https://rest.cbbanalytics.com'
API_VER = 'v3'  # v3 (recommended), v2, v1
BASE_URL = f'{HOST}/{API_VER}'
PAGING_MODE = 'cursor'  # cursor (recommended), offset
PAGE_SIZE = 1000  # the API max; fewer, larger pages cost fewer requests

# ids used by the examples; find others at /v3/competitions and /v3/conferences
COMPETITION_ID = 41097  # 2025-26 men's
ACC_CONFERENCE_ID = 53

# warn when the key hasn't been set
if API_KEY == 'YOUR_API_KEY_HERE':
    print('Set API_KEY above before running the requests below.')

## (B) Request helpers

In [ ]:
# one GET; reads the v3 envelope or a v1/v2 bare array, so the same code works on every version
def api_get(
    url,           # full endpoint url
    query_params   # dict of query params
):
    # returns {records, headers, meta}; meta is None on v1/v2

    # make the request, failing on any non-200
    res = requests.get(url, headers={'X-API-Key': API_KEY}, params=query_params, timeout=60)
    if res.status_code != 200:
        raise RuntimeError(f'API request failed: {res.status_code} {res.text}')

    # unwrap the v3 envelope; v1 and v2 return the records directly
    body = res.json()
    is_envelope = isinstance(body, dict) and 'data' in body.get('response', {})
    return {
        'records': body['response']['data'] if is_envelope else body,
        'headers': res.headers,
        'meta': body['response']['meta'] if is_envelope else None,
    }

In [ ]:
# page through every row with offset; slows down on deep pages, so prefer paging_cursor for big pulls
def paging_offset(
    url,            # full endpoint url
    query_params,   # dict of filters
    quiet=False     # True hides the per-page log
):
    # returns a list of records

    # request page after page, stopping at the first short page
    records, offset = [], 0
    while True:
        out = api_get(url, {**query_params, 'limit': PAGE_SIZE, 'offset': offset})
        records.extend(out['records'])
        if not quiet:
            print(f'  offset={offset}  records={len(out["records"])}')
        if len(out['records']) < PAGE_SIZE:
            break
        offset += PAGE_SIZE

    # return the records
    return records

In [ ]:
# page through every row with cursors; each page stays fast at any depth
def paging_cursor(
    url,            # full endpoint url
    query_params,   # dict of filters
    quiet=False     # True hides the per-page log
):
    # returns a list of records

    # request page after page, sending back each cursor until there isn't one
    records, after = [], None
    while True:
        out = api_get(url, {**query_params, 'limit': PAGE_SIZE, **({'after': after} if after else {})})
        records.extend(out['records'])
        if not quiet:
            print(f'  after={after or "(first page)"}  records={len(out["records"])}')

        # v3 puts the cursor in meta.nextCursor; v1/v2 in the X-Next-Cursor header
        after = (out['meta'] or {}).get('nextCursor') or out['headers'].get('X-Next-Cursor')
        if not after or not out['records']:
            break

    # return the records
    return records

## (C) `fetch_data`: every row of one endpoint as a DataFrame

In [ ]:
# fetch every row of an endpoint matching the filters, as a DataFrame
def fetch_data(
    table,              # path under the base url, e.g. 'stats/team/game-box'
    query_params=None,  # dict of filters, e.g. {'competitionIds': 41097}
    mode=PAGING_MODE,   # 'cursor' or 'offset'
    quiet=False         # True hides the per-page log
):
    # returns a DataFrame, one row per record; nested fields are flattened to dot-paths

    # page through the endpoint
    start_time = time.time()
    url = f'{BASE_URL}/{table}'
    if mode == 'cursor':
        records = paging_cursor(url, query_params or {}, quiet)
    elif mode == 'offset':
        records = paging_offset(url, query_params or {}, quiet)
    else:
        raise ValueError("mode must be 'cursor' or 'offset'")

    # build the frame, log the size and time, and return
    df = pd.json_normalize(records)
    if not quiet:
        print(f'  fetched {len(df)} rows x {len(df.columns)} cols from {table} in {time.time() - start_time:.1f}s')
    return df

## (D) Example fetches

### (D1) Entities: competitions, conferences, teams, players

In [ ]:
all_competitions = fetch_data('competitions')

In [ ]:
d1_conferences = fetch_data('conferences', {'divisionIds': 1})

In [ ]:
d1_teams = fetch_data('competition-teams', {'competitionIds': COMPETITION_ID, 'divisionIds': 1})

In [ ]:
acc_teams = fetch_data('competition-teams', {'competitionIds': COMPETITION_ID, 'conferenceIds': ACC_CONFERENCE_ID})

In [ ]:
d1_players = fetch_data('competition-team-players', {'competitionIds': COMPETITION_ID, 'divisionIds': 1})

In [ ]:
acc_players = fetch_data('competition-team-players', {'competitionIds': COMPETITION_ID, 'conferenceIds': ACC_CONFERENCE_ID})

### (D2) Team stats for all of D1

Always pass `splits` on aggregate endpoints; without it you get every split.

In [ ]:
team_game_box = fetch_data('stats/team/game-box', {'competitionIds': COMPETITION_ID})

In [ ]:
team_agg_box = fetch_data('stats/team/agg-box', {'competitionIds': COMPETITION_ID, 'splits': 'season', 'teamOrOpponent': 'team'})
team_agg_box.head()

In [ ]:
team_agg_pbp = fetch_data('stats/team/agg-pbp', {'competitionIds': COMPETITION_ID, 'splits': 'season', 'teamOrOpponent': 'team'})

### (D3) Player stats: game-level for the ACC, season totals for all of D1

In [ ]:
acc_player_game_box = fetch_data('stats/player/game-box', {'competitionIds': COMPETITION_ID, 'conferenceIds': ACC_CONFERENCE_ID})

In [ ]:
acc_player_game_pbp = fetch_data('stats/player/game-pbp', {'competitionIds': COMPETITION_ID, 'conferenceIds': ACC_CONFERENCE_ID})

In [ ]:
d1_player_agg_box = fetch_data('stats/player/agg-box', {'competitionIds': COMPETITION_ID, 'splits': 'season'})

### (D4) Rows changed since a date, for daily syncs

`updated=YYYY-MM-DD` returns rows changed on or after that UTC day. `PIPELINE_GUIDE.md` explains how to pick the date and how to store what comes back.

In [ ]:
since_date = (date.today() - timedelta(days=3)).isoformat()
recent_games = fetch_data('games', {'competitionIds': COMPETITION_ID, 'divisionIds': 1, 'updated': since_date})

In [ ]:
recent_team_game_box = fetch_data('stats/team/game-box', {'competitionIds': COMPETITION_ID, 'updated': since_date})

In [ ]:
recent_player_game_box = fetch_data('stats/player/game-box', {'competitionIds': COMPETITION_ID, 'updated': since_date})